# Mixing

This page builds an SIR model for three age groups and asks who infects whom. Two things change from the basic model:

- people **age**, moving from one age group to the next;
- the force of infection uses a **mixing matrix**, a table saying how much each age group mixes with each other age group.

We run the model twice, once with *proportionate* mixing (everybody mixes with everybody equally) and once with *assortative* mixing (people mostly mix with their own age group), and compare the infections in each age group.

Adapted from the [summer2 documentation](https://summer2.readthedocs.io) pages `examples/07-age-stratification` and `examples/09-mixing-matrices` at commit `d1537d6188aba85c33c0449b197eef6ad8b03d6c` of [monash-emu/summer2](https://github.com/monash-emu/summer2). Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried and adapted; code is written for summer4.

## Setting up the age groups

The disease states and the age groups are each a `Property`. Stratifying the states by age gives nine compartments: S, I and R for each of the three age groups. The population starts with 3,000 children, 5,000 adults and 2,000 elders, and ten infectious children.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import FlowModel, Param, Property, PropertyMap, TraitChain, TransitionFlow
from summer4.epi import ForceOfInfection, MixingMatrix

state = Property("state", ("S", "I", "R"))
age = Property("age", ("child", "adult", "elder"))
pmap = PropertyMap.from_property(state).stratify(age)
pmap.labels()

## Mixing matrices

A mixing matrix has one row and one column per age group. Row *a* says how much of the contact of age group *a* is spent with each age group. By default each row is rescaled to add up to one, so only the pattern in each row matters.

- **Proportionate** mixing: every row is the same, so everyone spreads their contact evenly across the age groups.
- **Assortative** mixing: the diagonal is large, so people mostly meet their own age group.

The next cell builds both matrices and draws each row as a stacked bar showing who that age group mixes with. In the proportionate plot every bar has the same three equal parts; in the assortative plot most of each bar is the person's own age group.

In [ ]:
proportionate = np.ones((3, 3))
assortative = np.array(
    [
        [0.8, 0.15, 0.05],
        [0.1, 0.8, 0.1],
        [0.05, 0.15, 0.8],
    ]
)

matrices = {"proportionate": proportionate, "assortative": assortative}
for name, matrix in matrices.items():
    shares = pd.DataFrame(matrix, index=age.traits, columns=age.traits)
    shares = shares.div(shares.sum(axis=1), axis=0)
    shares.plot.bar(
        barmode="stack",
        title=f"Who each age group mixes with: {name}",
        labels={"index": "age group of person", "value": "share of contact", "variable": "contacts with"},
    ).show()

## Building the model

The function below builds the model for any mixing matrix. It has three kinds of flow:

- **infection** moves susceptible people to infectious. The `ForceOfInfection` takes `group_by=age` and the `MixingMatrix`, so each age group feels a different force of infection;
- **recovery** moves infectious people to recovered;
- **ageing** moves people from child to adult and from adult to elder. A `TraitChain` lists the steps and the rate for each step (children spend about 15 years in their group, adults about 45).

All ten starting infections are children, so we can watch the infection spread to the other groups.

In [ ]:
def build_model(matrix):
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["I"],
            ForceOfInfection(
                "foi",
                infectious=state["I"],
                group_by=age,
                mixing=MixingMatrix(age, matrix),
                contact_rate=Param("contact_rate"),
            ),
        )
    )
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.2))
    model.add_flow(
        TransitionFlow(
            "ageing",
            age.present(),
            age.present(),
            1.0,
            pairing=TraitChain(
                age,
                (("child", "adult"), ("adult", "elder")),
                rates=(1.0 / 15.0, 1.0 / 45.0),
            ),
        )
    )
    model.set_initial_population(
        {
            state["S"] & age["child"]: 2990.0,
            state["I"] & age["child"]: 10.0,
            state["S"] & age["adult"]: 5000.0,
            state["S"] & age["elder"]: 2000.0,
        }
    )
    return model.compile()

## Running both models

Both runs use the same contact rate and cover 200 days. After each run we ask for the infection flow, summed within each age group with `sum_by(age, side="source")`. A flow connects two compartments, so you say whether to group by where the people came from (the source) or where they went. The plot shows new infections per day in each age group.

In [ ]:
params = {"contact_rate": 0.6}
infections = {}
for name, matrix in matrices.items():
    result = build_model(matrix).run(params, t0=0.0, t1=200.0, dt=1.0)
    by_age = result["infection"].sum_by(age, side="source")
    infections[name] = pd.DataFrame(
        np.asarray(by_age.values.data),
        index=pd.Index(np.asarray(by_age.times.values), name="day"),
        columns=age.traits,
    )

for name, frame in infections.items():
    frame.plot(
        title=f"New infections per day by age group: {name} mixing",
        labels={"value": "infections per day", "variable": "age group"},
    ).show()

## Comparing the two

To see the effect of the mixing pattern directly, the next cell adds up each age group's infections over the whole run and shows them as bars. The two kinds of mixing give different counts in each age group, even though the contact rate, the recovery rate and the ageing are identical.

In [ ]:
totals = pd.DataFrame({name: frame.sum() for name, frame in infections.items()})
totals.plot.bar(
    barmode="group",
    title="Total infections over 200 days by age group",
    labels={"index": "age group", "value": "people infected", "variable": "mixing"},
)

## Things to try

- Change `contact_rate` to 0.3 and 1.0. How do the two mixing patterns compare?
- Make the assortative matrix more extreme, with 0.95 on the diagonal. How late does the infection reach the elders?
- Turn ageing off by setting the rates in the `TraitChain` to `(0.0, 0.0)`.
- Seed the ten infections in the elders instead of the children.
- Pass `normalize="none"` to the `MixingMatrix` to use the matrix exactly as written, without rescaling the rows.